# Fraud-risk baseline

> **Dữ liệu synthetic:** kết quả chỉ phục vụ kiểm tra pipeline, không đại diện cho khách hàng thực.

Notebook này dùng lại hàm trong `src/`; đưa repository và `data/features/` lên Kaggle, rồi cập nhật `PROJECT_ROOT`. Các metric gồm RMSE, MAE, R², Spearman và precision/recall tại top-k trên validation.

In [1]:
from pathlib import Path
import sys
import yaml
import pandas as pd

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "configs" / "config.yaml").is_file():
    PROJECT_ROOT = PROJECT_ROOT.parent
if not (PROJECT_ROOT / "configs" / "config.yaml").is_file():
    raise FileNotFoundError("Không tìm thấy configs/config.yaml từ working directory hiện tại.")

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.models.train_fraud import train_fraud_baselines
from src.utils.io import read_parquet_safe

In [2]:
with (PROJECT_ROOT / 'configs/model_fraud.yaml').open(encoding='utf-8') as handle:
    fraud_config = yaml.safe_load(handle)

feature_dir = PROJECT_ROOT / 'data/features'
X_train = read_parquet_safe(feature_dir / 'X_train.parquet')
X_valid = read_parquet_safe(feature_dir / 'X_valid.parquet')
y_train = read_parquet_safe(feature_dir / 'y_fraud_train.parquet')['fraud_risk_score']
y_valid = read_parquet_safe(feature_dir / 'y_fraud_valid.parquet')['fraud_risk_score']

{"timestamp": "2026-11-06T13:15:13.739823+00:00", "level": "INFO", "logger": "src.utils.io", "message": "Reading Parquet: e:\\Personal Project\\Data science_analyze finance\\data\\features\\X_train.parquet"}
{"timestamp": "2026-11-06T13:15:14.457137+00:00", "level": "INFO", "logger": "src.utils.io", "message": "Reading Parquet: e:\\Personal Project\\Data science_analyze finance\\data\\features\\X_valid.parquet"}
{"timestamp": "2026-11-06T13:15:14.497553+00:00", "level": "INFO", "logger": "src.utils.io", "message": "Reading Parquet: e:\\Personal Project\\Data science_analyze finance\\data\\features\\y_fraud_train.parquet"}
{"timestamp": "2026-11-06T13:15:14.518406+00:00", "level": "INFO", "logger": "src.utils.io", "message": "Reading Parquet: e:\\Personal Project\\Data science_analyze finance\\data\\features\\y_fraud_valid.parquet"}


In [3]:
# Set log_mlflow=True when you want local Kaggle MLflow records under mlruns/.
fraud_results = train_fraud_baselines(
    X_train, y_train, X_valid, y_valid, fraud_config, seed=42, log_mlflow=False
)
pd.DataFrame({name: result.metrics for name, result in fraud_results.items()}).T

{"timestamp": "2026-11-04T02:34:37.490814+00:00", "level": "INFO", "logger": "src.utils.report", "message": "Wrote HTML report: reports\\fraud_baseline_dummy.html"}
{"timestamp": "2026-11-04T02:34:37.491712+00:00", "level": "INFO", "logger": "src.models.train_fraud", "message": "Fraud baseline dummy validation metrics: {'rmse': 0.15941378247536506, 'mae': 0.12159444944592596, 'r2': -1.211312848470314e-06, 'spearman': nan, 'precision_at_k': 0.04515555555555555, 'recall_at_k': 0.04515555555555555, 'k': 0.05, 'top_k_count': 5625.0}"}
{"timestamp": "2026-11-04T02:34:38.212662+00:00", "level": "INFO", "logger": "src.utils.report", "message": "Wrote HTML report: reports\\fraud_baseline_ridge.html"}
{"timestamp": "2026-11-04T02:34:38.213530+00:00", "level": "INFO", "logger": "src.models.train_fraud", "message": "Fraud baseline ridge validation metrics: {'rmse': 0.112264658471691, 'mae': 0.08554325786422685, 'r2': 0.5040534061364341, 'spearman': 0.6954532628136015, 'precision_at_k': 0.4624, 'r

,rmse,mae,r2,spearman,precision_at_k,recall_at_k,k,top_k_count
dummy,0.159414,0.121594,-0.000001,NaN,0.045156,0.045156,0.05,5625.0
ridge,0.112265,0.085543,0.504053,0.695453,0.462400,0.462400,0.05,5625.0


## Giai đoạn 6 — LightGBM

Các cell dưới đây dùng lại module trong `src/`. Chạy train trên Kaggle khi đã sẵn sàng dùng toàn bộ train/validation; không dùng test tại đây. Bundle versioned được tạo bởi `run_training(load_config())`, còn `final=True` chỉ dành cho lần đánh giá test đã chốt.

In [3]:
from src.models.train_fraud import train_fraud_lightgbm
from src.models.predict import load_model_bundle, predict_batch
from src.pipelines.training_pipeline import run_training
from src.utils.config import load_config

In [4]:
# Chạy riêng fraud LightGBM: Optuna chỉ dùng train, early stopping dùng validation.
fraud_lightgbm = train_fraud_lightgbm(
    X_train, y_train, X_valid, y_valid, fraud_config, seed=42
)
pd.DataFrame([fraud_lightgbm.metrics], index=['lightgbm'])

[I 2026-11-06 20:19:46,629] A new study created in memory with name: no-name-2de1b373-ff33-42c5-a7d3-68054c940e19
c:\Users\pc\AppData\Local\Programs\Python\Python313\Lib\site-packages\sklearn\utils\validation.py:2749: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(
[I 2026-11-06 20:19:52,893] Trial 0 finished with value: 0.10954720930836113 and parameters: {'num_leaves': 58, 'max_depth': 12, 'min_child_samples': 123, 'learning_rate': 0.09381218778758511, 'subsample': 0.7468055921327309, 'colsample_bytree': 0.7467983561008608, 'reg_alpha': 2.550264850403287e-06, 'reg_lambda': 1.156732719914599}. Best is trial 0 with value: 0.10954720930836113.
c:\Users\pc\AppData\Local\Programs\Python\Python313\Lib\site-packages\sklearn\utils\validation.py:2749: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(
[I 2026-11-06 20:19:56,909] Trial 1 finished with value: 0.1096

,rmse,mae,r2,spearman,precision_at_k,recall_at_k,k,top_k_count
lightgbm,0.109374,0.083564,0.529263,0.699825,0.477156,0.477156,0.05,5625.0


In [ ]:
# Sau khi chạy run_training(load_config()) để lưu bundle v1.0.0.
# Bundle inference nhận dữ liệu thô, không phải X_train đã transform.
fraud_bundle = load_model_bundle('v1.0.0', 'fraud_risk_score', PROJECT_ROOT / 'models')
raw_preview = pd.read_csv(PROJECT_ROOT / 'data/raw/synthetic_finance_20250901.csv', nrows=10)
predict_batch(raw_preview, fraud_bundle)